In [ ]:
import os
import zipfile
from pathlib import Path

input_root = Path('/kaggle/input')

def show_inputs():
    print('Mounted /kaggle/input entries:')
    for path in sorted(input_root.iterdir()):
        print(' -', path)
    print('\nZip files found:')
    for path in sorted(input_root.glob('**/*.zip')):
        print(' -', path)
    print('\nFirst 80 files/dirs under /kaggle/input:')
    for path in sorted(input_root.rglob('*'))[:80]:
        print(' -', path)

scripts = sorted(input_root.glob('**/aes2_submission.py'))

if not scripts:
    extract_root = Path('/kaggle/working/aes2_artifact')
    for zip_path in sorted(input_root.glob('**/*.zip')):
        with zipfile.ZipFile(zip_path) as archive:
            if any(name.endswith('aes2_submission.py') for name in archive.namelist()):
                archive.extractall(extract_root)
                os.environ['AES2_ARTIFACT_DIR'] = str(extract_root)
                scripts = sorted(extract_root.glob('**/aes2_submission.py'))
                break

if len(scripts) != 1:
    show_inputs()
    raise AssertionError(f'Expected exactly one aes2_submission.py, found: {scripts}')
os.environ['AES2_ARTIFACT_DIR'] = str(scripts[0].parent)
os.environ.setdefault('AES2_REQUIRE_CUDA', '1')
os.environ.setdefault('AES2_BATCH_SIZE', '32')
exec(compile(scripts[0].read_text(), str(scripts[0]), 'exec'))
